# RAG Fusion: Multi-Query Retrieval

A single query rarely expresses the full intent behind a question. RAG Fusion addresses this by generating several semantically diverse rewrites of the original query, retrieving documents for each, and merging the ranked result lists using **Reciprocal Rank Fusion (RRF)** — a rank aggregation method that rewards documents that appear highly across multiple lists without requiring score normalization.

**What you'll build:** A retrieval pipeline that turns one user query into N query variants, retrieves candidates for each, fuses the ranked lists, and returns a deduplicated, re-scored top-k to the LLM.

**Time:** ~20 min | **Difficulty:** Intermediate

## Prerequisites

- Completed the [Basic RAG](https://synapsekit.github.io/synapsekit-docs/guides/rag/) guide or equivalent experience
- `OPENAI_API_KEY` set in your environment

## What you'll learn

- How Reciprocal Rank Fusion works and why it beats score-based merging
- How `RAGFusionRetriever` generates and manages query variants
- How to tune the number of query variants and the RRF constant `k`
- How to inspect which documents were promoted or demoted by the fusion step

In [ ]:
!pip install synapsekit -q

## Environment Setup

Set your API key before running the notebook. In Colab, use the Secrets panel (key icon) or set it directly below.

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here or via environment

## Step 1: Install and configure

In [ ]:
import asyncio
import os

from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.retrievers import RAGFusionRetriever

llm = OpenAILLM(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings=embeddings)

## Step 2: Load your documents

In [ ]:
docs = [
    "Transformer models use self-attention to weigh token relationships across the full sequence.",
    "BERT pre-trains a bidirectional encoder using masked language modelling on large text corpora.",
    "GPT models generate text autoregressively, predicting each token from all preceding tokens.",
    "Retrieval-Augmented Generation combines a dense retriever with a generative language model.",
    "Attention mechanisms allow models to focus on the most relevant parts of the input dynamically.",
    "Fine-tuning adapts a pre-trained model to a downstream task using a smaller labelled dataset.",
    "Embeddings are dense vector representations that encode semantic meaning into a fixed-size space.",
    "Vector databases store embeddings and support approximate nearest-neighbour search at scale.",
]

await vectorstore.aadd(docs)
print(f"Added {len(docs)} documents to the vector store.")

## Step 3: Create the RAGFusionRetriever

The `num_queries` parameter controls how many variant queries are generated. More variants improve recall but increase LLM calls during query expansion.

In [ ]:
retriever = RAGFusionRetriever(
    vectorstore=vectorstore,
    llm=llm,
    num_queries=4,      # original + 3 rewrites
    top_k=5,            # documents returned after fusion
    rrf_k=60,           # RRF constant; higher values reduce the impact of rank differences
)
print("RAGFusionRetriever created with num_queries=4, top_k=5, rrf_k=60")

## Step 4: Inspect generated query variants

Before running a full pipeline, it helps to see the variants the LLM is generating. Poorly diverse variants are a sign the prompt needs tuning.

In [ ]:
async def inspect_variants():
    query = "How do language models process context?"
    variants = await retriever.agenerate_queries(query)
    for i, v in enumerate(variants):
        print(f"[{i}] {v}")

asyncio.run(inspect_variants())
# [0] How do language models process context?
# [1] What mechanisms do LLMs use to understand surrounding text?
# [2] How is context window handled in transformer architectures?
# [3] In what ways do neural language models capture long-range dependencies?

## Step 5: Retrieve with fusion

In [ ]:
async def fused_retrieve():
    query = "How do language models process context?"
    results = await retriever.aretrieve(query)
    for doc, score in results:
        print(f"[{score:.4f}] {doc[:80]}")

asyncio.run(fused_retrieve())

## Step 6: Wire fusion retrieval into a full RAG pipeline

In [ ]:
rag = RAG(
    llm=llm,
    retriever=retriever,
)
print("RAG pipeline assembled.")

## Step 7: Ask questions and stream the answer

In [ ]:
async def ask(question: str):
    print(f"Q: {question}\n")
    async for chunk in rag.astream(question):
        print(chunk, end="", flush=True)
    print()

asyncio.run(ask("How do language models process context?"))

## Step 8: Verify fusion improved recall on an ambiguous query

Run the same query through a plain vector retriever and the fusion retriever side by side to see which documents change rank.

In [ ]:
from synapsekit.retrievers import VectorRetriever

plain = VectorRetriever(vectorstore=vectorstore, top_k=5)

async def compare(query: str):
    plain_results = await plain.aretrieve(query)
    fused_results = await retriever.aretrieve(query)

    plain_ids = {doc for doc, _ in plain_results}
    fused_ids = {doc for doc, _ in fused_results}

    print("Only in fused:", fused_ids - plain_ids)
    print("Only in plain:", plain_ids - fused_ids)

asyncio.run(compare("How do language models process context?"))

## Complete Working Example

A single self-contained cell that runs the entire RAG Fusion pipeline end-to-end.

In [ ]:
import asyncio
import os

from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.retrievers import RAGFusionRetriever

DOCS = [
    "Transformer models use self-attention to weigh token relationships across the full sequence.",
    "BERT pre-trains a bidirectional encoder using masked language modelling on large text corpora.",
    "GPT models generate text autoregressively, predicting each token from all preceding tokens.",
    "Retrieval-Augmented Generation combines a dense retriever with a generative language model.",
    "Attention mechanisms allow models to focus on the most relevant parts of the input dynamically.",
    "Fine-tuning adapts a pre-trained model to a downstream task using a smaller labelled dataset.",
    "Embeddings are dense vector representations that encode semantic meaning into a fixed-size space.",
    "Vector databases store embeddings and support approximate nearest-neighbour search at scale.",
]


async def main():
    llm = OpenAILLM(model="gpt-4o-mini")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings=embeddings)

    await vectorstore.aadd(DOCS)

    retriever = RAGFusionRetriever(
        vectorstore=vectorstore,
        llm=llm,
        num_queries=4,
        top_k=5,
        rrf_k=60,
    )

    rag = RAG(llm=llm, retriever=retriever)

    question = "How do language models process context?"
    print(f"Q: {question}\n")
    async for chunk in rag.astream(question):
        print(chunk, end="", flush=True)
    print()


asyncio.run(main())

## Next Steps

- [Self-RAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/self-rag) — grade retrieved documents for relevance before sending them to the LLM
- [Cross-Encoder Reranking](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/cross-encoder-reranking) — combine fusion with precise cross-encoder scoring for the highest-quality top-k
- [Query Decomposition](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/query-decomposition) — instead of rephrasing, break a complex question into independent sub-queries